# Gold — dim_tiempo

Genera la dimensión temporal para el star schema. No lee Silver — es un catálogo sintético.

| Columna | Tipo | Descripción |
|---|---|---|
| `tiempo_sk` | INT | Surrogate key |
| `anio` | INT | Año (2012–2025) |
| `periodo_covid` | STRING | `pre_covid` (≤2019) \| `covid_y_post` (≥2020) |

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

GOLD_SCHEMA = 'gold_ss2'
WRITE_MODE  = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Construcción de dim_tiempo

Cubre todos los años con datos en el DW: 2012–2025 (rango de las fuentes MSPAS, INE y WHO).

In [0]:
rows = [
    (y, 'pre_covid' if y <= 2019 else 'covid_y_post')
    for y in range(2012, 2026)
]

df_dim = spark.createDataFrame(rows, ['anio', 'periodo_covid'])

df_dim = df_dim.withColumn(
    'tiempo_sk',
    F.row_number().over(Window.orderBy('anio'))
)

df_dim = df_dim.select('tiempo_sk', 'anio', 'periodo_covid')

logger.info(f'dim_tiempo: {df_dim.count()} filas')

df_dim.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.dim_tiempo')

logger.info(f'Escrito → {GOLD_SCHEMA}.dim_tiempo')

## Validación

In [0]:
df_val = spark.table(f'{GOLD_SCHEMA}.dim_tiempo')
print(f'Total filas: {df_val.count()}')
df_val.show(20)

print('\n── Verificar rango de años ──')
df_val.agg(F.min('anio').alias('min_anio'), F.max('anio').alias('max_anio')).show()

print('\n── Distribución por periodo_covid ──')
df_val.groupBy('periodo_covid').count().orderBy('periodo_covid').show()